# House Price Prediction using Linear Regression
**MDI3003 — Advanced Predictive Analytics  Lab 01**

**Student Name:** Harshita Bogineni

**Reg No:** 23MID0043

**Dataset:** California Housing

## 1. Imports

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, platform, joblib
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split,KFold,cross_validate,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,PolynomialFeatures
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression,Ridge,Lasso,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
SEED=42

## 2. Environment

In [2]:
print(platform.python_version())

3.11.0


## 3. Load Dataset

In [3]:
housing=fetch_california_housing(as_frame=True)
df=housing.frame.rename(columns={'MedHouseVal':'Price'})
display(df.head())

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  Longitude  Price
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88    -122.23  4.526
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86    -122.22  3.585
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85    -122.24  3.521
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85    -122.25  3.413
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85    -122.25  3.422


## 4. Data Audit

In [4]:
display(df.shape)
display(df.info())
display(df.describe().T)
display(df.isna().sum())
display(df.duplicated().sum())

(20640, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
 8   Price       20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None
              count         mean          std         min         25%          50%          75%           max
MedInc      20640.0     3.870671     1.899822    0.499900    2.563400     3.534800     4.743250     15.000100
HouseAge    20640.0    28.639486    12.585558    1.000000   18.000000    29.000000    37.000000     52.000000
AveRooms    20640.0     5.429000     2.474173    0.8461

## 5. EDA

In [5]:
sns.histplot(df['Price'],kde=True); plt.show()
plt.figure(figsize=(10,8)); sns.heatmap(df.corr(),cmap='coolwarm'); plt.show()
for col in ['MedInc','HouseAge','AveRooms','AveBedrms','Population']:
    sns.scatterplot(data=df,x=col,y='Price'); plt.show()
for c in df.columns:
    sns.boxplot(x=df[c]); plt.title(c); plt.show()

## 6. Split

In [6]:
X=df.drop(columns='Price'); y=df['Price']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=SEED)

## 7. Preprocessing

In [7]:
num=X.select_dtypes(include=np.number).columns
cat=X.select_dtypes(exclude=np.number).columns
pre=ColumnTransformer([
('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),
('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)
])

## 8. Baseline

In [8]:
d=DummyRegressor()
d.fit(X_train[num],y_train)
p=d.predict(X_test[num])
mae=mean_absolute_error(y_test,p)
mse=mean_squared_error(y_test,p)
rmse=np.sqrt(mse)
r2=r2_score(y_test,p)
print('Baseline MAE:',mae)
print('Baseline MSE:',mse)
print('Baseline RMSE:',rmse)
print('Baseline R2:',r2)


Baseline MAE: 0.9060685490007149
Baseline MSE: 1.3106960720039365
Baseline RMSE: 1.1448563543099792
Baseline R2: -0.00021908714592466794


## 9. Simple Linear Regression

In [9]:
slr=LinearRegression()
slr.fit(X_train[['MedInc']],y_train)
pred=slr.predict(X_test[['MedInc']])
print(r2_score(y_test,pred))

0.45885918903846656


## 10. Multiple Models

In [10]:
models={
'Linear Regression':LinearRegression(),
'Ridge':Ridge(),
'Lasso':Lasso(),
'ElasticNet':ElasticNet(),
'Decision Tree':DecisionTreeRegressor(random_state=SEED),
'Random Forest':RandomForestRegressor(random_state=SEED),
'Gradient Boosting':GradientBoostingRegressor(random_state=SEED)}
results=[]; fitted={}
for n,m in models.items():
 pipe=Pipeline([('pre',pre),('model',m)])
 pipe.fit(X_train,y_train)
 pr=pipe.predict(X_test)
 mae=mean_absolute_error(y_test,pr)
 mse=mean_squared_error(y_test,pr)
 rmse=np.sqrt(mse)
 r2=r2_score(y_test,pr)
 Xt=pipe.named_steps['pre'].transform(X_test)
 n_samples,n_features=Xt.shape
 adj_r2=np.nan if n_samples<=n_features+1 else 1-(1-r2)*(n_samples-1)/(n_samples-n_features-1)
 results.append([n,mae,mse,rmse,r2,adj_r2])
 fitted[n]=pipe
results_df=pd.DataFrame(results,columns=['Model','MAE','MSE','RMSE','R2','Adjusted_R2']).sort_values('RMSE')
display(results_df)


               Model       MAE       MSE      RMSE        R2  Adjusted_R2
5      Random Forest  0.327425  0.255170  0.505143  0.805275     0.804897
6  Gradient Boosting  0.371650  0.293999  0.542217  0.775643     0.775208
4      Decision Tree  0.453904  0.493969  0.702829  0.623042     0.622310
1              Ridge  0.533193  0.555855  0.745557  0.575816     0.574992
0  Linear Regression  0.533200  0.555892  0.745581  0.575788     0.574964
3         ElasticNet  0.805995  1.044231  1.021876  0.203126     0.201578
2              Lasso  0.906069  1.310696  1.144856 -0.000219    -0.002162


## 11. Polynomial Regression

In [11]:
poly=Pipeline([
('imp',SimpleImputer(strategy='median')),
('poly',PolynomialFeatures(degree=2,include_bias=False)),
('scale',StandardScaler()),
('model',Ridge(alpha=1.0))
])
poly.fit(X_train[['MedInc','HouseAge','AveRooms']],y_train)
print(poly.score(X_test[['MedInc','HouseAge','AveRooms']],y_test))

0.5100776029076949


## 12. Cross Validation

In [12]:
cv=KFold(n_splits=5,shuffle=True,random_state=SEED)
rows=[]
for n,m in models.items():
 pipe=Pipeline([('pre',pre),('model',m)])
 s=cross_validate(pipe,X_train,y_train,cv=cv,scoring=['neg_root_mean_squared_error','r2'])
 rows.append([n,-s['test_neg_root_mean_squared_error'].mean(),s['test_r2'].mean()])
cv_df=pd.DataFrame(rows,columns=['Model','CV_RMSE','CV_R2'])
display(cv_df)

               Model   CV_RMSE     CV_R2
0  Linear Regression  0.720510  0.611457
1              Ridge  0.720509  0.611457
2              Lasso  1.156172 -0.000215
3         ElasticNet  1.028845  0.207961
4      Decision Tree  0.733307  0.597257
5      Random Forest  0.510928  0.804650
6  Gradient Boosting  0.532138  0.788084


## 13. Hyperparameter Tuning

In [13]:
grid=GridSearchCV(Pipeline([('pre',pre),('model',Ridge())]),
{'model__alpha':[0.001,0.01,0.1,1,10,100]},cv=5,
scoring='neg_root_mean_squared_error')
grid.fit(X_train,y_train)
print(grid.best_params_, -grid.best_score_)

{'model__alpha': 0.001} 0.7205271876569361


## 14. Residual Analysis

In [14]:
best=fitted[results_df.iloc[0]['Model']]
pred=best.predict(X_test)
res=y_test-pred
plt.scatter(pred,res); plt.axhline(0,color='r'); plt.show()
sns.histplot(res,kde=True); plt.show()
plt.scatter(y_test,pred)
plt.plot([y_test.min(),y_test.max()],[y_test.min(),y_test.max()],'r--')
plt.show()

## 15. Feature Importance

In [15]:
if hasattr(best.named_steps['model'],'feature_importances_'):
    fi=pd.DataFrame({'Feature':best.named_steps['pre'].get_feature_names_out(),
                     'Importance':best.named_steps['model'].feature_importances_}).sort_values('Importance',ascending=False)
    display(fi.head(20))
elif hasattr(best.named_steps['model'],'coef_'):
    coef=pd.DataFrame({'Feature':best.named_steps['pre'].get_feature_names_out(),
                       'Coefficient':best.named_steps['model'].coef_})
    display(coef.head())

           Feature  Importance
0      num__MedInc    0.524871
5    num__AveOccup    0.138443
6    num__Latitude    0.088936
7   num__Longitude    0.088629
1    num__HouseAge    0.054593
2    num__AveRooms    0.044272
4  num__Population    0.030650
3   num__AveBedrms    0.029606


## 16. Save Outputs

In [19]:
results_df.to_csv('California_model_comparison.csv',index=False)
cv_df.to_csv('California_cross_validation_results.csv',index=False)
joblib.dump(best,'California_house_price_pipeline.joblib')
